In [ ]:
import os
import random

import torch
import torchvision.datasets as dset
import torchvision.transforms as transforms
from matplotlib import pyplot as plt
from torch.distributions import Bernoulli

os.chdir("..")  # Run in project directory

# Deep Reinforcement Learning _in Action_
## MNIST Genetic Algorithm

Setup a directory to store the MNIST dataset/

In [ ]:
root = "./data"
if not os.path.exists(root):
    print("Data directory missing, creating directory...")
    os.mkdir(root)

Setup a transformer to normalize the data.

In [ ]:
trans = transforms.Compose(
    [transforms.ToTensor(), transforms.Normalize((0.5,), (1.0,))]
)

In [ ]:
train_set = dset.MNIST(root=root, train=True, transform=trans, download=True)
test_set = dset.MNIST(root=root, train=False, transform=trans, download=True)

In [ ]:
batch_size = 100

train_loader = torch.utils.data.DataLoader(
    dataset=train_set, batch_size=batch_size, shuffle=True
)
test_loader = torch.utils.data.DataLoader(
    dataset=test_set, batch_size=batch_size, shuffle=False
)

We define a simple linear classifier (or you can think of it as a single layer neural network). It simply multiplies a weight/parameter matrix by the input vector and applies a softmax.

In [ ]:
x = next(iter(train_loader))[0]

In [ ]:
x = x.reshape(batch_size, 784)

In [ ]:
class Individual:
    def __init__(self, param, fitness=0):
        self.param = param
        self.fitness = fitness

In [ ]:
def model(x, W):
    return x @ W  # CrossEntropyLoss consumes logits, not probabilities.

In [ ]:
model(x, torch.rand(784, 10))

In [ ]:
def spawn_population(param_size=(784, 10), pop_size=1000):
    return [Individual(torch.randn(*param_size)) for i in range(pop_size)]

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss()

In [ ]:
random.randint(0, 10)

In [ ]:
def evaluate_population(pop):
    avg_fit = 0  # avg population fitness
    x, y = next(iter(train_loader))
    for individual in pop:
        pred = model(x.reshape(batch_size, 784), individual.param)
        loss = loss_fn(pred, y)
        fit = loss
        individual.fitness = 1.0 / fit
        avg_fit += fit
    avg_fit = avg_fit / len(pop)
    return pop, avg_fit

In [ ]:
def recombine(x1, x2):  # x1,x2 : Individual
    w1 = x1.param.view(-1)  # flatten
    w2 = x2.param.view(-1)
    cross_pt = random.randint(0, w1.shape[0])
    child1 = torch.zeros(w1.shape)
    child2 = torch.zeros(w1.shape)
    child1[0:cross_pt] = w1[0:cross_pt]
    child1[cross_pt:] = w2[cross_pt:]
    child2[0:cross_pt] = w2[0:cross_pt]
    child2[cross_pt:] = w1[cross_pt:]
    child1 = child1.reshape(784, 10)
    child2 = child2.reshape(784, 10)
    c1 = Individual(child1)
    c2 = Individual(child2)
    return [c1, c2]

In [ ]:
def mutate(pop, mut_rate=0.01):
    param_shape = pop[0].param.shape
    l = torch.zeros(*param_shape)
    l[:] = mut_rate
    m = Bernoulli(l)
    for individual in pop:
        mut_vector = m.sample() * torch.randn(*param_shape)
        individual.param = mut_vector + individual.param
    return pop

In [ ]:
def seed_next_population(pop, pop_size=1000, mut_rate=0.01):
    new_pop = []
    while len(new_pop) < pop_size:  # until new pop is full
        parents = random.choices(pop, k=2, weights=[x.fitness for x in pop])
        offspring = recombine(parents[0], parents[1])
        new_pop.extend(offspring)
    new_pop = mutate(new_pop, mut_rate)
    return new_pop

In [ ]:
pop = spawn_population()

In [ ]:
%%time
pop, avg_fit = evaluate_population(pop)

In [ ]:
new_pop = seed_next_population(pop)

In [ ]:
len(new_pop)

Now we need to spawn a population of weight matrices, run the model using the different individuals, calculate the loss for each one, and then breed the ones with the highest fitness score (lowest loss).

In [ ]:
num_generations = 50
population_size = 100
mutation_rate = 0.001

### Main Evolution (Training) Loop

In [ ]:
pop_fit = []
pop = spawn_population(pop_size=population_size)  # initial population
for gen in range(num_generations):
    # trainning
    pop, avg_fit = evaluate_population(pop)
    pop_fit.append(avg_fit)  # record population average fitness
    new_pop = seed_next_population(
        pop, pop_size=population_size, mut_rate=mutation_rate
    )
    pop = new_pop

In [ ]:
plt.plot(pop_fit)

In [ ]:
avg_loss = 0
for i in range(len(pop)):
    x, y = next(iter(train_loader))
    pred = model(x.reshape(batch_size, 784), pop[i].param)
    loss = loss_fn(pred, y)
    avg_loss += loss
avg_loss /= len(pop)
print(avg_loss)

Avg Loss new pop: 2.3336
Avg Loss after 10 gens: 2.3435

In [ ]:
pop, _ = evaluate_population(pop)
pop.sort(key=lambda x: x.fitness, reverse=True)
print(pop[0].fitness)

In [ ]:
def model_accuracy(dataset, model_params):
    correct, total = 0, 0
    for x, y in dataset:
        pred = model(x.reshape(batch_size, 784), model_params)
        pred = pred.argmax(dim=1)

        correct += (pred == y).sum().item()
        total += y.size(0)

    accuracy = correct / total
    print(f"Accuracy: {accuracy:.2%}")


model_accuracy(train_loader, pop[0].param)
model_accuracy(test_loader, pop[0].param)

## Train with gradient-descent (comparison)

In [ ]:
p = torch.randn(784, 10, requires_grad=True)
optim = torch.optim.Adam(lr=1e-3, params=[p])

In [ ]:
loss_list = []
for epoch in range(50):
    for x, y in train_loader:
        optim.zero_grad()
        pred = model(x.reshape(batch_size, 784), p)
        loss = loss_fn(pred, y)

        loss.backward()
        optim.step()

        loss_list.append(loss.item())

    print(f"Epoch {epoch+1}: loss = {loss.item():.4f}")

In [ ]:
plt.plot(loss_list)

In [ ]:
model_accuracy(train_loader, p)
model_accuracy(test_loader, p)